
# Bokeh for Time Series Analysis
<hr style="border: 2px solid black;">


<img src="./images/bokeh.png" alt="bokeh Logo" width="1000"/>
<hr style="border: 2px solid black;">

<img src="./images/bokeh_at_ag_glance.png" alt="bokeh Logo" width="1000"/>
<hr style="border: 2px solid black;">
**Introduction to Bokeh**
Bokeh is an interactive visualization library for Python that targets modern web browsers for presentation.
Unlike Matplotlib, which is primarily designed for static plots, Bokeh excels at creating
interactive plots and dashboards. It can handle large datasets and streaming data,
making it suitable for real-time applications.

**Key Features of Bokeh:**

* **Interactivity:** Built-in support for zooming, panning, hovering, and other interactive tools.
* **Web-Focused:** Generates HTML and JavaScript, making it easy to embed plots in web pages.
* **High Performance:** Can handle large datasets efficiently.
* **Versatility:** Supports a wide range of plot types (lines, bars, scatter plots, etc.).

<hr style="border: 2px solid black;">


**Documentation:**

For comprehensive documentation, please refer to the official Bokeh website: [https://docs.bokeh.org/en/latest/](https://docs.bokeh.org/en/latest/)


<hr style="border: 2px solid black;">


**Lab Exercise:**

Your task is to recreate the time series analysis lab we previously conducted using Pandas,
Matplotlib, and Seaborn, but this time, utilize the Bokeh library for visualization.
This will involve:

1.  Loading and preprocessing the "AirPassengersDates.csv" dataset.
2.  Creating interactive Bokeh plots for:
    * Time series line plots
    * Bar plots of aggregated data
    * Visualizing mean and standard deviation
    * Outlier detection
    * Resampling (upsampling and downsampling)
    * Lag analysis
    * Autocorrelation

Pay close attention to Bokeh's features for interactivity (tools, hover effects) and
its handling of data sources. Aim to replicate the insights and visualizations
from the previous lab while leveraging Bokeh's strengths.

Good luck!
<hr style="border: 2px solid black;">

In [12]:
import pandas as pd
import numpy as np
import os

# --- ÉTAPE 0 : Chargement robuste ---
file_path = 'AirPassengersDates.csv'

if os.path.exists(file_path):
    df = pd.read_csv(file_path, parse_dates=['Month'])
    print(f"Fichier '{file_path}' chargé avec succès.")
else:
    print(f"INFO : '{file_path}' introuvable. Génération de données de simulation...")
    date_rng = pd.date_range(start='1949-01-01', end='1960-12-01', freq='MS')
    t = np.arange(len(date_rng))
    passengers = 100 + 2.5 * t + 20 * np.sin(2 * np.pi * t / 12) + np.random.normal(0, 10, len(date_rng))
    df = pd.DataFrame({'Month': date_rng, '#Passengers': passengers.astype(int)})

df.set_index('Month', inplace=True)

# --- ÉTAPE 1 : Moyenne mobile & Outliers ---
df['rolling_mean'] = df['#Passengers'].rolling(window=12, center=True).mean()
df['rolling_std'] = df['#Passengers'].rolling(window=12, center=True).std()

# Détection (2.5 sigma)
df['upper'] = df['rolling_mean'] + (2.5 * df['rolling_std'])
df['lower'] = df['rolling_mean'] - (2.5 * df['rolling_std'])
df['is_outlier'] = (df['#Passengers'] > df['upper']) | (df['#Passengers'] < df['lower'])

# --- ÉTAPE 2 : Rééchantillonnage (CORRECTIF ICI) ---
# 'A' fonctionne sur les anciennes versions, 'YE' sur les très récentes. 
# On utilise 'A' pour la compatibilité maximale.
df_annual = df['#Passengers'].resample('A').mean().reset_index()

# --- ÉTAPE 3 : Lag ---
df['lag_1'] = df['#Passengers'].shift(1)
autocorr_val = df['#Passengers'].autocorr(lag=1)

print("Données prêtes !")

INFO : 'AirPassengersDates.csv' introuvable. Génération de données de simulation...
Données prêtes !


In [14]:
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.layouts import column, row

# Active l'affichage dans le Notebook (Jupyter/Colab)
try:
    output_notebook()
except:
    print("Mode script standard détecté.")

# Source de données
source = ColumnDataSource(df.reset_index())
source_annual = ColumnDataSource(df_annual)
outliers_df = df[df['is_outlier'] == True].reset_index()
source_outliers = ColumnDataSource(outliers_df)

# 1. Main Plot
p1 = figure(x_axis_type='datetime', title="Série Temporelle : Passagers & Outliers", 
           width=800, height=350, tools="pan,wheel_zoom,box_zoom,reset")

p1.line('Month', '#Passengers', source=source, color="gray", alpha=0.3, legend_label="Brut")
p1.line('Month', 'rolling_mean', source=source, color="#1f77b4", line_width=3, legend_label="Moyenne 12m")
p1.scatter('Month', '#Passengers', source=source_outliers, color="red", size=8, legend_label="Outliers")

p1.add_tools(HoverTool(tooltips=[("Date", "@Month{%F}"), ("Valeur", "@#Passengers")],
                       formatters={'@Month': 'datetime'}))

# 2. Bar Chart (Annuel)
p2 = figure(x_axis_type='datetime', title="Moyenne par Année", width=400, height=300)
p2.vbar(x='Month', top='#Passengers', width=pd.Timedelta(days=150), source=source_annual, color="orange")

# 3. Lag Plot
p3 = figure(title=f"Autocorrélation (Lag 1): {autocorr_val:.2f}", width=400, height=300)
p3.scatter('lag_1', '#Passengers', source=source, alpha=0.5, color="green")

# Layout
layout = column(p1, row(p2, p3))
show(layout)

Loading BokehJS ...